# Reproducing Manuscript Tables and Figures (Tier 1)

This notebook reproduces all tables, statistical tests, sensitivity audits, and publication figures for:

> **Global forecasts can close most of the flood warning gap**

All computations execute deterministically on CPU from the companion Zenodo derived benchmark archive (`10.5281/zenodo.24583418` — *pre-reserved archive DOI; unpublished draft pending final release*). Zero proprietary credentials, raw CAP alert texts, or third-party raw geometry archives are required.

### Prerequisites
1. Install the package with notebook support: `pip install -e "..[notebooks]"` (or `pip install -e ".[notebooks]"` from the repository root).
2. Download and unpack the companion Zenodo archive, then set `CAP_GAP_ZENODO_DIR` in your environment (or edit `ZENODO_DIR` below).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

from cap_alerting_coverage.cli import (
    PARITY_TABLES,
    PASS_THROUGH_TAGS,
    VALIDATION_PARITY_TABLES,
    compare_csv_tables,
)
from cap_alerting_coverage.config import ProjectPaths
from cap_alerting_coverage.manuscript_outputs import generate_manuscript_tables_and_figures
from cap_alerting_coverage.schemas import (
    validate_cap_gid0_matrix,
    validate_cap_gid1_matrix,
    validate_deidentified_validation_table,
    validate_event_catalog,
    validate_ffm_gid1_matrix,
    validate_tier_cube,
)
from cap_alerting_coverage.scoring import run_full_scoring
from cap_alerting_coverage.validation_audit import (
    compute_confusion_matrix_table,
    compute_spatial_precision_table,
    load_deidentified_validation_table,
)
from cap_alerting_coverage.zenodo_fetch import verify_zenodo_checksums

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
ZENODO_DIR = Path(os.environ.get("CAP_GAP_ZENODO_DIR", str(REPO_ROOT / "zenodo_archive"))).expanduser().resolve()
RESULTS_DIR = Path(os.environ.get("CAP_GAP_RESULTS_DIR", str(REPO_ROOT / "out_results"))).expanduser().resolve()
FIGURES_DIR = Path(os.environ.get("CAP_GAP_FIGURES_DIR", str(REPO_ROOT / "out_figures"))).expanduser().resolve()

paths = ProjectPaths.from_env(
    repo_root=REPO_ROOT,
    zenodo_dir=ZENODO_DIR,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
)
print("Configured ProjectPaths successfully (Zenodo dir exists:", paths.zenodo_dir.is_dir(), ")")

## 1. Verify Zenodo Archive Checksums & Schema / Privacy Contracts

Check `SHA256SUMS.txt` and verify that every derived input catalog, Admin-1 exceedance matrix, discrete return-period tier cube, and de-identified `N = 400` validation file satisfies strict schema and non-redistribution guardrails.

In [ ]:
if (paths.zenodo_dir / "SHA256SUMS.txt").exists():
    verified = verify_zenodo_checksums(paths.zenodo_dir)
    print(f"SHA-256 verified {len(verified)} files in staged Zenodo archive.")
else:
    print("SHA256SUMS.txt not found; skipping checksum verification.")

validate_event_catalog(
    pd.read_parquet(paths.resolve_data_file("catalog_without_gs_all_185_2021_2025.parquet", "01_event_catalogs")),
    dataset_name="Catalog A (185)",
)
validate_event_catalog(
    pd.read_parquet(paths.resolve_data_file("catalog_with_gs_all_185_2021_2025.parquet", "01_event_catalogs")),
    dataset_name="Catalog B (185)",
)
validate_cap_gid1_matrix(
    pd.read_parquet(paths.resolve_data_file("cap_gid1_daily_2020_2025.parquet", "02_admin1_exceedance_matrices"))
)
validate_cap_gid0_matrix(
    pd.read_parquet(paths.resolve_data_file("cap_gid0_daily_2020_2025.parquet", "02_admin1_exceedance_matrices"))
)
validate_ffm_gid1_matrix(
    pd.read_parquet(paths.resolve_data_file("ffm_gid1_daily_2021_2025.parquet", "02_admin1_exceedance_matrices"))
)

dhm_npz = dict(np.load(paths.resolve_data_file("dhm_10d_gid1_multilead_2021_2025.npz", "02_admin1_exceedance_matrices"), allow_pickle=True))
validate_tier_cube(dhm_npz, expected_ndim=3, gid1_list=dhm_npz["gid_1"], dataset_name="DHM_10D")
print(f"All derived schemas & privacy checks passed ({len(dhm_npz['gid_1']):,} Admin-1 units).")

## 2. Run Full Tier-1 Benchmark Scoring, Tables, and Figures

In [ ]:
scoring_state = run_full_scoring(paths, strict_185=True)
outputs = generate_manuscript_tables_and_figures(
    paths,
    scoring_state=scoring_state,
    strict_185=True,
    render_figures=True,
)
print(f"Generated {len(outputs['figures'])} publication figures (300 DPI).")

ref_dir = paths.zenodo_dir / "03_summary_results"
val_ref_dir = paths.zenodo_dir / "04_validation_deidentified"
if ref_dir.is_dir():
    rows = []
    for fname in PARITY_TABLES:
        ref_p = ref_dir / fname
        act_p = paths.results_dir / fname
        ok, msg = compare_csv_tables(act_p, ref_p, atol=1e-3)
        tag = PASS_THROUGH_TAGS.get(fname, "").strip(" []") or "RECOMPUTED"
        rows.append({"table": fname, "mode": tag, "status": "PASS" if ok else "FAIL", "detail": msg})
    if val_ref_dir.is_dir():
        for fname in VALIDATION_PARITY_TABLES:
            ref_p = val_ref_dir / fname
            act_p = paths.results_dir / fname
            if ref_p.exists() and act_p.exists():
                ok, msg = compare_csv_tables(act_p, ref_p, atol=1e-3)
                rows.append({"table": fname, "mode": "VALIDATION", "status": "PASS" if ok else "FAIL", "detail": msg})
    parity_df = pd.DataFrame(rows)
    display(parity_df)
    assert (parity_df["status"] == "PASS").all(), "One or more parity checks failed!"
else:
    print("Reference 03_summary_results/ directory not present; skipping parity diff.")

In [ ]:
ref_dir = paths.zenodo_dir / "03_summary_results"
if ref_dir.is_dir():
    rows = []
    for fname in PARITY_TABLES:
        ref_p = ref_dir / fname
        act_p = paths.results_dir / fname
        if ref_p.exists() and act_p.exists():
            ok, msg = compare_csv_tables(act_p, ref_p, atol=1e-3)
            rows.append({"table": fname, "status": "PASS" if ok else "FAIL", "detail": msg})
    parity_df = pd.DataFrame(rows)
    display(parity_df)
    assert (parity_df["status"] == "PASS").all(), "One or more parity checks failed!"
else:
    print("Reference 03_summary_results/ directory not present; skipping parity diff.")

## 4. Inspect Manuscript Tables (Tables 1–2 & Extended Data Tables 1, 3–6)

In [ ]:
print("=== Table 1: Headline Coverage & Hybrid Gap Closure ===")
display(pd.read_csv(paths.results_dir / "table1_summary.csv"))

if (paths.results_dir / "table2_gauge_vs_catalogs.csv").exists():
    print("=== Table 2: Stream-Gauge Exceedance Precision vs. Disaster Catalogs ===")
    display(pd.read_csv(paths.results_dir / "table2_gauge_vs_catalogs.csv"))

print("=== Extended Data Table 3: Sensitivity Audit Matrix ===")
display(pd.read_csv(paths.results_dir / "ed_table3_sensitivity_matrix.csv"))

print("=== Extended Data Table 6a: Annual CAP Onboarding Trajectory (2021-2025) ===")
display(pd.read_csv(paths.results_dir / "ed_table6a_annual_onboarding.csv"))

## 5. De-identified N = 400 CAP Keyword Validation Audit

In [ ]:
val_path = paths.zenodo_dir / "04_validation_deidentified" / "validation_n400_deidentified.csv"
if val_path.exists():
    val_df = load_deidentified_validation_table(val_path)
    validate_deidentified_validation_table(val_df)
    cm = compute_confusion_matrix_table(val_df)
    sp = compute_spatial_precision_table(val_df)
    print("=== Spatial Pairing Precision (subset = all) ===")
    display(sp[sp["subset"] == "all"])
    print("=== Thematic Confusion Matrices (subset = all, v3 vs v4_reviewed) ===")
    display(cm[(cm["subset"] == "all") & (cm["label_version"].isin(["v3", "v4_reviewed"]))])
else:
    display(pd.read_csv(paths.results_dir / "ed_table4_validation_summary.csv"))

## 6. Display Main and Extended Data Figures

In [ ]:
import matplotlib.image as mpimg

figure_order = [
    "fig-1a.png",
    "fig-1b.png",
    "fig-2a.png",
    "fig-2b.png",
    "fig-2c.png",
    "fig-2_composite.png",
    "fig-3.png",
    "fig-4.png",
    "ed-fig-1.png",
    "ed-fig-2.png",
    "ed-fig-3.png",
    "ed-fig-4.png",
    "ed-fig-5.png",
]
inline_preview = os.environ.get("CAP_GAP_NB_INLINE_IMAGES", "0") == "1"
fig_rows = []
for fig_name in figure_order:
    fig_p = paths.figures_dir / fig_name
    assert fig_p.exists(), f"Missing expected figure: {fig_name}"
    arr = mpimg.imread(fig_p)
    fig_rows.append({
        "figure": fig_name,
        "width_px": int(arr.shape[1]),
        "height_px": int(arr.shape[0]),
        "size_kb": round(fig_p.stat().st_size / 1024.0, 1),
    })
    if inline_preview:
        display(Image(filename=str(fig_p), width=800))
display(pd.DataFrame(fig_rows))